# Fine-tune SpeechBrain UrbanSound8K (ECAPA) on Uganda 61K subset (8 classes)

This notebook fine-tunes a pretrained SpeechBrain UrbanSound8K ECAPA classifier on a selected subset of 8 classes from the Uganda 61K dataset and evaluates the results.

Target classes:

- motorvehicle-horn (2)
- motorvehicle-siren (4)
- car-alarm (5)
- community-radio (8)
- construction-site (11)
- mobile-music (6)
- generator (13)
- crowd-noise (16)

In [10]:
# Environment and imports
import os, sys, json, math, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# Prefer local SpeechBrain from repo if available to avoid network installs
base_dir = Path().resolve()  # directory of this notebook (UrbanNoiseUganda61K)
repo_audio_classifier_dir = 'D:/Projects/UrbanNoiseClassifier/project/audio-classifier'
local_speechbrain_path = 'D:/Projects/UrbanNoiseClassifier/project/audio-classifier/speechbrain-classifier/speechbrain'
if Path(local_speechbrain_path).exists():
    # Add .../speechbrain-classifier to sys.path so that `import speechbrain` works
    sys.path.insert(0, str(Path(local_speechbrain_path).parent))
    try:
        import speechbrain  # noqa: F401
        print('Using local SpeechBrain at', Path(local_speechbrain_path).parent)
    except Exception as e:
        print('Local SpeechBrain import failed, will try pip if needed:', e)
else:
    print('Local SpeechBrain path not found, will try pip install in the next cell if needed.')

print('Python:', sys.version)
print('Torch :', torch.__version__)
print('CUDA :', torch.cuda.is_available())
if torch.cuda.is_available():
    try:
        print('GPU  :', torch.cuda.get_device_name(0))
    except Exception:
        pass
    torch.backends.cudnn.benchmark = True

Using local SpeechBrain at D:\Projects\UrbanNoiseClassifier\project\audio-classifier\speechbrain-classifier
Python: 3.13.9 (tags/v3.13.9:8183fa5, Oct 14 2025, 14:09:13) [MSC v.1944 64 bit (AMD64)]
Torch : 2.8.0+cu126
CUDA : True
GPU  : NVIDIA GeForce RTX 3050 Laptop GPU


In [11]:
# If SpeechBrain isn't available yet, install it (internet required).
try:
    import speechbrain  # noqa: F401
    print('SpeechBrain already available')
except Exception:
    import subprocess, sys as _sys
    print('Installing SpeechBrain...')
    _ = subprocess.check_call([_sys.executable, '-m', 'pip', 'install', 'speechbrain', 'scikit-learn', 'torchaudio'])
    import speechbrain  # noqa: F401
    print('SpeechBrain installed')

SpeechBrain already available


## Paths and configuration

In [12]:
# Data prepared by loadAndVerify.ipynb (filtered 8 classes only)
filtered_dir = Path('D:/Projects/UrbanNoiseClassifier/project/audio-classifier/data/raw/UrbanNoiseUganda61K/filtered_large_audio')
labels_csv = filtered_dir / 'filtered_labels.csv'
assert filtered_dir.exists(), f'Filtered audio folder not found: {filtered_dir}. Run the export cell in loadAndVerify.ipynb.'
assert labels_csv.exists(), f'Filtered labels CSV not found: {labels_csv}. Run the export cell in loadAndVerify.ipynb.'

# Pretrained model directory from repo
pretrained_dir = Path(repo_audio_classifier_dir) / Path('speechbrain-classifier/pretrained_models/urbansound8k_ecapa')
assert pretrained_dir.exists(), f'Pretrained directory not found: {pretrained_dir}'

# Fine-tuned output directory
finetune_out = Path(repo_audio_classifier_dir) / 'speechbrain-classifier' / 'pretrained_models' / 'urbansound8k_ecapa_subset8_finetuned'
finetune_out.mkdir(parents=True, exist_ok=True)

# Audio / training configuration
target_sr = 16000  # ECAPA typically uses 16k
num_epochs_head = 10  # train classifier head
num_epochs_ft = 5     # optional light fine-tuning of embedding model
batch_size = 64       # try a larger batch to better utilize GPU memory; tune if OOM
lr_head = 1e-3
lr_ft = 1e-4
# Use more loader workers to keep GPU fed; Windows benefits from persistent workers
cpu_cores = os.cpu_count() or 4
num_workers = 0 # max(2, min(8, cpu_cores // 2))
persistent_workers = True if num_workers > 0 else False
prefetch_factor = 4 if num_workers > 0 else None
seed = 42
device = torch.device('cpu') # ('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed) if torch.cuda.is_available() else None
print('filtered_dir:', filtered_dir)
print('labels_csv  :', labels_csv)
print('pretrained  :', pretrained_dir)
print('output      :', finetune_out)
print('device      :', device)
print('batch_size  :', batch_size)
print('num_workers :', num_workers, '| persistent_workers:', persistent_workers, '| prefetch_factor:', prefetch_factor)

filtered_dir: D:\Projects\UrbanNoiseClassifier\project\audio-classifier\data\raw\UrbanNoiseUganda61K\filtered_large_audio
labels_csv  : D:\Projects\UrbanNoiseClassifier\project\audio-classifier\data\raw\UrbanNoiseUganda61K\filtered_large_audio\filtered_labels.csv
pretrained  : D:\Projects\UrbanNoiseClassifier\project\audio-classifier\speechbrain-classifier\pretrained_models\urbansound8k_ecapa
output      : D:\Projects\UrbanNoiseClassifier\project\audio-classifier\speechbrain-classifier\pretrained_models\urbansound8k_ecapa_subset8_finetuned
device      : cpu
batch_size  : 64
num_workers : 0 | persistent_workers: False | prefetch_factor: None


## Load labels and build splits

In [13]:
df = pd.read_csv(labels_csv)
assert {'filename','class','class_id','split'} <= set(df.columns), 'filtered_labels.csv must have filename, class, class_id, split'
classes = sorted(df['class'].unique().tolist())
num_classes = len(classes)
print('Detected classes (from CSV):', classes)
print('num_classes =', num_classes)
assert num_classes == 8, 'Expected 8 classes in filtered dataset'

# Label encoders
cls_to_idx = {c:i for i,c in enumerate(classes)}
idx_to_cls = {i:c for c,i in cls_to_idx.items()}

# Build a stratified split using filenames and labels (ignore provided split if you prefer custom %)
X = df['filename'].values
y = df['class'].map(cls_to_idx).values
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=seed)
train_idx, test_idx = next(sss1.split(X, y))
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.1765, random_state=seed)  # ~0.15 of total for val (0.15/0.85 ≈ 0.1765)
train_idx2, val_idx = next(sss2.split(X_train, y_train))
X_train, X_val = X_train[train_idx2], X_train[val_idx]
y_train, y_val = y_train[train_idx2], y_train[val_idx]
print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')

# Persist splits for reproducibility
splits_path = finetune_out / 'splits.json'
with open(splits_path, 'w') as f:
    json.dump({
        'classes': classes,
        'train': X_train.tolist(),
        'val': X_val.tolist(),
        'test': X_test.tolist()
    }, f, indent=2)
print('Saved splits to', splits_path)

Detected classes (from CSV): ['car-alarm', 'community-radio', 'construction-site', 'crowd-noise', 'generator', 'mobile-music', 'motorvehicle-horn', 'motorvehicle-siren']
num_classes = 8
Train: 13326  Val: 2857  Test: 2856
Saved splits to D:\Projects\UrbanNoiseClassifier\project\audio-classifier\speechbrain-classifier\pretrained_models\urbansound8k_ecapa_subset8_finetuned\splits.json


## Dataset and DataLoaders

In [14]:
class SoundDataset(Dataset):
    def __init__(self, root: Path, filenames, labels, target_sr=16000, augment=False, use_cache=True):
        self.root = Path(root)
        self.filenames = list(filenames)
        self.labels = list(labels)
        self.target_sr = target_sr
        self.augment = augment
        self.use_cache = use_cache
        self.resamplers = {}  # cache per source sr
        self._cache = {}  # cache preprocessed waveforms
        
        # Preload and cache all audio files (non-augmented) if caching is enabled
        if self.use_cache:
            print(f'Preloading and caching {len(self.filenames)} audio files...')
            from tqdm.auto import tqdm
            for idx in tqdm(range(len(self.filenames)), desc='Caching audio'):
                self._load_and_cache(idx)
            print(f'Cached {len(self._cache)} audio files')

    def __len__(self):
        return len(self.filenames)

    def _resample(self, wav, sr):
        if sr == self.target_sr:
            return wav
        if sr not in self.resamplers:
            self.resamplers[sr] = torchaudio.transforms.Resample(sr, self.target_sr)
        return self.resamplers[sr](wav)

    def _augment(self, wav):
        # Simple augmentation: additive noise and random gain
        if not self.augment:
            return wav
        if random.random() < 0.5:
            gain_db = random.uniform(-6, 6)
            wav = wav * (10 ** (gain_db / 20.0))
        if random.random() < 0.3:
            noise = torch.randn_like(wav) * 0.005
            wav = wav + noise
        return wav

    def _load_and_cache(self, idx):
        """Load, preprocess (mono conversion + resample), and cache a single audio file."""
        if idx in self._cache:
            return
        
        fn = self.filenames[idx]
        path = self.root / fn
        wav, sr = torchaudio.load(str(path))  # [channels, time]
        if wav.dim() == 2 and wav.size(0) > 1:
            wav = wav.mean(dim=0, keepdim=True)  # mono
        if wav.dim() == 1:
            wav = wav.unsqueeze(0)
        wav = self._resample(wav, sr)
        wav = wav.squeeze(0)  # [time]
        self._cache[idx] = wav

    def __getitem__(self, idx):
        label = int(self.labels[idx])
        
        if self.use_cache:
            # Retrieve from cache
            wav = self._cache[idx].clone()  # clone to avoid modifying cached version
        else:
            # Load on-the-fly (original behavior)
            fn = self.filenames[idx]
            path = self.root / fn
            wav, sr = torchaudio.load(str(path))  # [channels, time]
            if wav.dim() == 2 and wav.size(0) > 1:
                wav = wav.mean(dim=0, keepdim=True)  # mono
            if wav.dim() == 1:
                wav = wav.unsqueeze(0)
            wav = self._resample(wav, sr)
            wav = wav.squeeze(0)  # [time]
        
        # Apply augmentation (only affects training set with augment=True)
        wav = self._augment(wav)
        return wav, label


In [18]:
# SOLUTION 1: Disable caching when using multiple workers on Windows
# Caching causes massive overhead due to Windows 'spawn' multiprocessing
# which must pickle/unpickle the entire cache for each worker

use_caching = False  # Set to True only if num_workers == 0 or on Linux
print(f"Audio caching: {'enabled' if use_caching else 'disabled'} (recommended: disabled for Windows + multi-worker)")

train_ds = SoundDataset(filtered_dir, X_train, y_train, target_sr=target_sr, augment=True, use_cache=use_caching)
val_ds   = SoundDataset(filtered_dir, X_val,   y_val,   target_sr=target_sr, augment=False, use_cache=use_caching)
test_ds  = SoundDataset(filtered_dir, X_test,  y_test,  target_sr=target_sr, augment=False, use_cache=use_caching)

Audio caching: disabled (recommended: disabled for Windows + multi-worker)


### Alternative Solutions if Benchmark Still Slow

If the dataloader benchmark is still slow, try these additional optimizations:

In [17]:
# SOLUTION 2: Reduce workers for faster initialization on Windows
# Try num_workers=0 or num_workers=2 to reduce spawn overhead

# Temporarily override for testing
test_num_workers = 0  # or try 2
test_loader_kwargs = dict(batch_size=batch_size, pin_memory=True, num_workers=test_num_workers)

train_loader_fast = DataLoader(train_ds, shuffle=True, **test_loader_kwargs)
print(f"Created loader with num_workers={test_num_workers} (vs original {num_workers})")

# Quick test
import time
print("Testing first batch...")
t0 = time.perf_counter()
batch = next(iter(train_loader_fast))
t1 = time.perf_counter()
print(f"First batch time: {t1-t0:.2f}s | batch shapes: {batch[0].shape}, {batch[1].shape}")

Created loader with num_workers=0 (vs original 0)
Testing first batch...


d:\Projects\UrbanNoiseClassifier\.venv\Lib\site-packages\torchaudio\_backend\utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(


RuntimeError: stack expects each tensor to be equal size, but got [168960] at entry 0 and [178560] at entry 2

In [19]:
train_ds = SoundDataset(filtered_dir, X_train, y_train, target_sr=target_sr, augment=True)
val_ds   = SoundDataset(filtered_dir, X_val,   y_val,   target_sr=target_sr, augment=False)
test_ds  = SoundDataset(filtered_dir, X_test,  y_test,  target_sr=target_sr, augment=False)

loader_kwargs = dict(batch_size=batch_size, pin_memory=True)
if num_workers and num_workers > 0:
    loader_kwargs.update(dict(num_workers=num_workers, persistent_workers=persistent_workers, prefetch_factor=prefetch_factor))

train_loader = DataLoader(train_ds, shuffle=True,  **loader_kwargs)
val_loader   = DataLoader(val_ds,   shuffle=False, **loader_kwargs)
test_loader  = DataLoader(test_ds,  shuffle=False, **loader_kwargs)

len(train_ds), len(val_ds), len(test_ds)

Preloading and caching 13326 audio files...


Caching audio:   0%|          | 0/13326 [00:00<?, ?it/s]

Cached 13326 audio files
Preloading and caching 2857 audio files...


Caching audio:   0%|          | 0/2857 [00:00<?, ?it/s]

Cached 2857 audio files
Preloading and caching 2856 audio files...


Caching audio:   0%|          | 0/2856 [00:00<?, ?it/s]

Cached 2856 audio files


(13326, 2857, 2856)

In [20]:
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("CUDA device name:", torch.cuda.get_device_name(0))


CUDA available: True
CUDA device count: 1
CUDA device name: NVIDIA GeForce RTX 3050 Laptop GPU


## Load pretrained SpeechBrain model and replace classifier head

In [21]:
# Import EncoderClassifier with compatibility across SpeechBrain versions
try:
    from speechbrain.inference.classifiers import EncoderClassifier
except Exception:
    from speechbrain.pretrained import EncoderClassifier

# Try to load from local savedir (expects hyperparams.yaml and ckpts in the folder)
enc = EncoderClassifier.from_hparams(source=str(pretrained_dir), savedir=str(pretrained_dir))
# Make sure the encoder lives on the chosen device
enc = enc.to(device)
try:
    enc.device = device
except Exception:
    pass
enc.eval()
print(enc)
print('Encoder device =', device)

D:\Projects\UrbanNoiseClassifier\project\audio-classifier\speechbrain-classifier\speechbrain\speechbrain\utils\torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()
D:\Projects\UrbanNoiseClassifier\project\audio-classifier\speechbrain-classifier\speechbrain\speechbrain\utils\torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and vide

EncoderClassifier(
  (mods): ModuleDict(
    (compute_features): Fbank(
      (compute_STFT): STFT()
      (compute_fbanks): Filterbank()
      (compute_deltas): Deltas()
      (context_window): ContextWindow()
    )
    (mean_var_norm): InputNormalization()
    (embedding_model): ECAPA_TDNN(
      (blocks): ModuleList(
        (0): TDNNBlock(
          (conv): Conv1d(
            (conv): Conv1d(80, 1024, kernel_size=(5,), stride=(1,))
          )
          (activation): ReLU()
          (norm): BatchNorm1d(
            (norm): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          )
          (dropout): Dropout1d(p=0.0, inplace=False)
        )
        (1): SERes2NetBlock(
          (tdnn1): TDNNBlock(
            (conv): Conv1d(
              (conv): Conv1d(1024, 1024, kernel_size=(1,), stride=(1,))
            )
            (activation): ReLU()
            (norm): BatchNorm1d(
              (norm): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affin

In [22]:
# Inspect embedding dimension by running a dummy forward to get embeddings
with torch.no_grad():
    # Ensure encoder is on the selected device
    enc = enc.to(device)
    try:
        enc.device = device
    except Exception:
        pass
    wav0, _ = train_ds[0]
    wav0 = wav0.unsqueeze(0).to(device)  # [1, time]
    emb = enc.encode_batch(wav0)  # shape [1, 1, emb_dim] or [1, emb_dim]
    emb = emb.squeeze()
    emb_dim = emb.shape[-1]
print('Embedding dim =', emb_dim, '| device =', device)

Embedding dim = 192 | device = cpu


In [23]:
# Build a new classifier head
new_head = nn.Sequential(
    nn.Linear(emb_dim, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, num_classes)
)
# Place the head on the same device as the encoder
try:
    head_device = enc.device
except Exception:
    head_device = device
new_head = new_head.to(head_device)

# Freeze embedding model parameters; we'll train the new head first
for p in enc.mods.embedding_model.parameters():
    p.requires_grad = False

# Combine into a simple wrapper for training
class SBWrapper(nn.Module):
    def __init__(self, encoder, head):
        super().__init__()
        self.encoder = encoder
        self.head = head
    def forward(self, wav_batch):  # wav_batch: [B, T]
        with torch.no_grad():
            emb = self.encoder.encode_batch(wav_batch)  # [B, 1, D] or [B, D]
        emb = emb.squeeze(1) if emb.dim() == 3 else emb
        return self.head(emb)  # [B, C]

# Create wrapper and ensure it lives on the same device as encoder
model = SBWrapper(enc, new_head)
model = model.to(head_device)
print(model)
print('Wrapper model device =', head_device)

SBWrapper(
  (encoder): EncoderClassifier(
    (mods): ModuleDict(
      (compute_features): Fbank(
        (compute_STFT): STFT()
        (compute_fbanks): Filterbank()
        (compute_deltas): Deltas()
        (context_window): ContextWindow()
      )
      (mean_var_norm): InputNormalization()
      (embedding_model): ECAPA_TDNN(
        (blocks): ModuleList(
          (0): TDNNBlock(
            (conv): Conv1d(
              (conv): Conv1d(80, 1024, kernel_size=(5,), stride=(1,))
            )
            (activation): ReLU()
            (norm): BatchNorm1d(
              (norm): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            )
            (dropout): Dropout1d(p=0.0, inplace=False)
          )
          (1): SERes2NetBlock(
            (tdnn1): TDNNBlock(
              (conv): Conv1d(
                (conv): Conv1d(1024, 1024, kernel_size=(1,), stride=(1,))
              )
              (activation): ReLU()
              (norm): Batch

In [24]:
# Quick DataLoader benchmark: measure time-to-first-batch for current and a minimal loader
import time
print(f"train_loader batches: {len(train_loader)}")
t0 = time.perf_counter()
batch = next(iter(train_loader))
t1 = time.perf_counter()
print(f"Current loader: first batch time = {t1 - t0:.2f}s; shapes = {batch[0].shape}, {batch[1].shape}")

# Minimal loader to isolate worker/prefetch overhead
quick_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0, pin_memory=False)
t0 = time.perf_counter()
qbatch = next(iter(quick_loader))
t1 = time.perf_counter()
print(f"Quick loader (workers=0, bs=8): first batch time = {t1 - t0:.2f}s; shapes = {qbatch[0].shape}, {qbatch[1].shape}")

# Show torchaudio backend for context
try:
    print('torchaudio backend:', torchaudio.get_audio_backend())
except Exception as e:
    print('torchaudio backend query failed:', e)

train_loader batches: 209


RuntimeError: stack expects each tensor to be equal size, but got [183360] at entry 0 and [175680] at entry 1

## Train classifier head

In [ ]:
optimizer = torch.optim.Adam(new_head.parameters(), lr=lr_head)
criterion = nn.CrossEntropyLoss()
best_val_acc = 0.0
best_head_path = finetune_out / 'best_head.pt'

# Use encoder's device for consistency
_run_device = enc.device if hasattr(enc, 'device') else device
use_amp = (_run_device.type == 'cuda')
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

for epoch in range(1, num_epochs_head+1):
    print(f'--- Epoch {epoch}/{num_epochs_head} ---')
    model.train()
    total, correct, running_loss = 0, 0, 0.0
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs_head} [train]", dynamic_ncols=True, leave=False)
    for wav, lbl in train_bar:
        wav = wav.to(_run_device, non_blocking=True)
        lbl = lbl.to(_run_device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if use_amp:
            with torch.cuda.amp.autocast():
                logits = model(wav)
                loss = criterion(logits, lbl)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(wav)
            loss = criterion(logits, lbl)
            loss.backward()
            optimizer.step()
        running_loss += loss.item() * wav.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == lbl).sum().item()
        total += wav.size(0)
        train_bar.set_postfix({
            'loss': f"{running_loss / max(1,total):.4f}",
            'acc': f"{(correct / max(1,total)):.4f}"
        })
    train_acc = correct / max(1, total)
    train_loss = running_loss / max(1, total)

    # Validation
    model.eval()
    v_total, v_correct = 0, 0
    val_bar = tqdm(val_loader, desc=f"Epoch {epoch}/{num_epochs_head} [val]", dynamic_ncols=True, leave=False)
    with torch.no_grad():
        for wav, lbl in val_bar:
            wav = wav.to(_run_device, non_blocking=True)
            lbl = lbl.to(_run_device, non_blocking=True)
            if use_amp:
                with torch.cuda.amp.autocast():
                    logits = model(wav)
            else:
                logits = model(wav)
            pred = logits.argmax(dim=1)
            v_correct += (pred == lbl).sum().item()
            v_total += wav.size(0)
            val_bar.set_postfix({
                'acc': f"{(v_correct / max(1,v_total)):.4f}"
            })
    val_acc = v_correct / max(1, v_total)
    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} val_acc={val_acc:.4f}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(new_head.state_dict(), best_head_path)
        print('  -> saved new best head to', best_head_path)

C:\Users\prana\AppData\Local\Temp\ipykernel_16648\281382329.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(_run_device.type == 'cuda'))


--- Epoch 1/10 ---


## Optional: light fine-tuning of embedding model

In [ ]:
# Load best head first
if best_head_path.exists():
    new_head.load_state_dict(torch.load(best_head_path, map_location=device))

# Unfreeze last layers of the embedding model (if available) for a few epochs
for p in enc.mods.embedding_model.parameters():
    p.requires_grad = True

# You can optionally freeze earlier layers by name if model exposes them
# for name, module in enc.mods.embedding_model.named_children():
#     print(name)  # inspect and choose to freeze some layers if desired

# Make sure encoder and head are on the same device
ft_device = enc.device if hasattr(enc, 'device') else device
enc = enc.to(ft_device)
new_head = new_head.to(ft_device)

optimizer_ft = torch.optim.Adam(list(enc.mods.embedding_model.parameters()) + list(new_head.parameters()), lr=lr_ft)
best_val_acc_ft = best_val_acc
best_full_path = finetune_out / 'best_full_finetuned.pt'

scaler_ft = torch.cuda.amp.GradScaler(enabled=(ft_device.type == 'cuda'))

for epoch in range(1, num_epochs_ft+1):
    enc.train(); new_head.train()
    total, correct, running_loss = 0, 0, 0.0
    train_bar = tqdm(train_loader, desc=f"[FT] Epoch {epoch}/{num_epochs_ft} [train]", dynamic_ncols=True, leave=False)
    for wav, lbl in train_bar:
        wav = wav.to(ft_device, non_blocking=True)
        lbl = lbl.to(ft_device, non_blocking=True)
        optimizer_ft.zero_grad(set_to_none=True)
        # forward pass with trainable encoder (AMP enabled)
        with torch.cuda.amp.autocast(enabled=(ft_device.type == 'cuda')):
            emb = enc.encode_batch(wav)  # [B, 1, D]
            emb = emb.squeeze(1) if emb.dim() == 3 else emb
            logits = new_head(emb)
            loss = criterion(logits, lbl)
        scaler_ft.scale(loss).backward()
        scaler_ft.step(optimizer_ft)
        scaler_ft.update()
        running_loss += loss.item() * wav.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == lbl).sum().item()
        total += wav.size(0)
        train_bar.set_postfix({
            'loss': f"{running_loss / max(1,total):.4f}",
            'acc': f"{(correct / max(1,total)):.4f}"
        })
    train_acc = correct / max(1, total)
    train_loss = running_loss / max(1, total)

    # Validation
    enc.eval(); new_head.eval()
    v_total, v_correct = 0, 0
    val_bar = tqdm(val_loader, desc=f"[FT] Epoch {epoch}/{num_epochs_ft} [val]", dynamic_ncols=True, leave=False)
    with torch.no_grad():
        for wav, lbl in val_bar:
            wav = wav.to(ft_device, non_blocking=True)
            lbl = lbl.to(ft_device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=(ft_device.type == 'cuda')):
                emb = enc.encode_batch(wav)
                emb = emb.squeeze(1) if emb.dim() == 3 else emb
                logits = new_head(emb)
            pred = logits.argmax(dim=1)
            v_correct += (pred == lbl).sum().item()
            v_total += wav.size(0)
            val_bar.set_postfix({
                'acc': f"{(v_correct / max(1,v_total)):.4f}"
            })
    val_acc = v_correct / max(1, v_total)
    print(f"[FT] Epoch {epoch:02d} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} val_acc={val_acc:.4f}")
    if val_acc > best_val_acc_ft:
        best_val_acc_ft = val_acc
        torch.save({
            'encoder_state': enc.state_dict(),
            'head_state': new_head.state_dict(),
            'classes': classes
        }, best_full_path)
        print('  -> saved new best full model to', best_full_path)

## Evaluation on test set

In [ ]:
# Load best available weights (prefer fully fine-tuned if present)
full_ckpt = finetune_out / 'best_full_finetuned.pt'
if full_ckpt.exists():
    ckpt = torch.load(full_ckpt, map_location=device)
    enc.load_state_dict(ckpt['encoder_state'])
    new_head.load_state_dict(ckpt['head_state'])
    print('Loaded fully fine-tuned weights')
elif best_head_path.exists():
    new_head.load_state_dict(torch.load(best_head_path, map_location=device))
    print('Loaded best head-only weights')
else:
    print('No saved weights found; evaluating current model state')

# Ensure everything is on the same device
_eval_device = enc.device if hasattr(enc, 'device') else device
enc = enc.to(_eval_device)
new_head = new_head.to(_eval_device)

enc.eval(); new_head.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for wav, lbl in test_loader:
        wav = wav.to(_eval_device)
        lbl = lbl.to(_eval_device)
        emb = enc.encode_batch(wav)
        emb = emb.squeeze(1) if emb.dim() == 3 else emb
        logits = new_head(emb)
        pred = logits.argmax(dim=1)
        y_true.extend(lbl.cpu().numpy().tolist())
        y_pred.extend(pred.cpu().numpy().tolist())

acc = accuracy_score(y_true, y_pred)
print(f'Test accuracy: {acc:.4f}')
report = classification_report(y_true, y_pred, target_names=[idx_to_cls[i] for i in range(num_classes)], digits=4)
print(report)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[idx_to_cls[i] for i in range(num_classes)], yticklabels=[idx_to_cls[i] for i in range(num_classes)])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

# Save metrics
metrics_path = finetune_out / 'metrics.json'
with open(metrics_path, 'w') as f:
    json.dump({'test_accuracy': float(acc), 'classification_report': report}, f, indent=2)
print('Saved metrics to', metrics_path)

## Save final artifacts and quick inference helper

In [ ]:
# Save label mapping and head weights for easy reuse
label_map_path = finetune_out / 'label_mapping.json'
with open(label_map_path, 'w') as f:
    json.dump({'idx_to_cls': idx_to_cls, 'cls_to_idx': cls_to_idx}, f, indent=2)
print('Saved label mapping to', label_map_path)

final_head_path = finetune_out / 'final_head.pt'
torch.save(new_head.state_dict(), final_head_path)
print('Saved final head to', final_head_path)

In [ ]:
# Quick inference helper: classify a single WAV from filtered_dir
def classify_file(wav_path):
    # Use the same device as the encoder
    infer_device = enc.device if hasattr(enc, 'device') else device
    enc.eval(); new_head.eval()
    wav, sr = torchaudio.load(str(wav_path))
    if wav.dim() == 2 and wav.size(0) > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if wav.dim() == 1:
        wav = wav.unsqueeze(0)
    if sr != target_sr:
        wav = torchaudio.transforms.Resample(sr, target_sr)(wav)
    wav = wav.squeeze(0).to(infer_device)
    with torch.no_grad():
        emb = enc.encode_batch(wav.unsqueeze(0))
        emb = emb.squeeze(1) if emb.dim() == 3 else emb
        logits = new_head(emb)
        probs = F.softmax(logits, dim=1).cpu().numpy()[0]
        pred_idx = int(np.argmax(probs))
        return idx_to_cls[pred_idx], float(probs[pred_idx])

# Example usage: pick a random test file
sample_fn = random.choice(list(X_test))
pred_label, confidence = classify_file(filtered_dir / sample_fn)
print('Sample:', sample_fn, '->', pred_label, f'({confidence:.2%})')

In [ ]:
# Utility: ensure encoder, head, and wrapper are on the same device
try:
    _target_device = device
except NameError:
    _target_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def sync_devices(target_device=_target_device):
    global enc, new_head, model, device
    device = torch.device(target_device)
    if 'enc' in globals():
        try:
            enc = enc.to(device)
            # Some SpeechBrain versions track .device separately
            try:
                enc.device = device
            except Exception:
                pass
        except Exception as e:
            print('Warning: could not move encoder to device:', e)
    if 'new_head' in globals():
        try:
            new_head = new_head.to(device)
        except Exception as e:
            print('Warning: could not move head to device:', e)
    if 'model' in globals():
        try:
            model = model.to(device)
        except Exception as e:
            print('Warning: could not move wrapper model to device:', e)
    print('Synchronized modules to device =', device)

# Call once after any device change
sync_devices(device)

In [ ]:
# Optional: I/O + resample micro-benchmark to spot bottlenecks
import time, statistics as stats
sample_files = random.sample(list(X_train), min(12, len(X_train)))
load_times, resample_times = [], []
for fn in sample_files:
    p = filtered_dir / fn
    t0 = time.perf_counter()
    wav, sr = torchaudio.load(str(p))
    t1 = time.perf_counter()
    if wav.dim() == 2 and wav.size(0) > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if wav.dim() == 1:
        wav = wav.unsqueeze(0)
    if sr != target_sr:
        resampler = torchaudio.transforms.Resample(sr, target_sr)
        wav = resampler(wav)
    t2 = time.perf_counter()
    load_times.append(t1 - t0)
    resample_times.append(max(0.0, t2 - t1))
print(f"I/O load    avg={stats.mean(load_times):.03f}s  median={stats.median(load_times):.03f}s over {len(load_times)} files")
print(f"Resample    avg={stats.mean(resample_times):.03f}s  median={stats.median(resample_times):.03f}s over {len(resample_times)} files")